# Calendar Dimension Table

Builds the standard calendar dimension `careplus_db_ws.silver.calender` with one row per day from 1940-01-01 to 2030-12-31, including date key, year, quarter, month, week, day and flag attributes.

In [0]:
%sql
CREATE OR REPLACE TABLE careplus_db_ws.silver.calender
COMMENT 'Standard calendar dimension, one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)                AS date_key,
  d                                                      AS date,
  -- Year
  year(d)                                                AS year,
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END              AS half_year,
  CASE WHEN (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0
       THEN true ELSE false END                          AS is_leap_year,
  -- Quarter
  quarter(d)                                             AS quarter,
  concat('Q', quarter(d))                                AS quarter_name,
  concat(year(d), '-Q', quarter(d))                      AS year_quarter,
  -- Month
  month(d)                                               AS month,
  date_format(d, 'MMMM')                                 AS month_name,
  date_format(d, 'MMM')                                  AS month_short_name,
  date_format(d, 'yyyy-MM')                              AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                  AS year_month_key,
  day(last_day(d))                                       AS days_in_month,
  -- Week (ISO 8601, weeks start Monday)
  weekofyear(d)                                          AS week_of_year,
  extract(YEAROFWEEK FROM d)                             AS iso_year,
  concat(extract(YEAROFWEEK FROM d), '-W', lpad(weekofyear(d), 2, '0')) AS year_week,
  CAST(ceil(day(d) / 7.0) AS INT)                        AS week_of_month,
  to_date(date_trunc('WEEK', d))                         AS week_start_date,
  date_add(to_date(date_trunc('WEEK', d)), 6)            AS week_end_date,
  -- Day
  day(d)                                                 AS day_of_month,
  dayofyear(d)                                           AS day_of_year,
  extract(DOW_ISO FROM d)                                AS day_of_week,        -- 1 = Monday ... 7 = Sunday
  date_format(d, 'EEEE')                                 AS day_name,
  date_format(d, 'EEE')                                  AS day_short_name,
  -- Period boundaries
  to_date(date_trunc('MONTH', d))                        AS month_start_date,
  last_day(d)                                            AS month_end_date,
  to_date(date_trunc('QUARTER', d))                      AS quarter_start_date,
  date_sub(add_months(to_date(date_trunc('QUARTER', d)), 3), 1) AS quarter_end_date,
  to_date(date_trunc('YEAR', d))                         AS year_start_date,
  make_date(year(d), 12, 31)                             AS year_end_date,
  -- Flags
  extract(DOW_ISO FROM d) IN (6, 7)                      AS is_weekend,
  extract(DOW_ISO FROM d) NOT IN (6, 7)                  AS is_weekday,
  d = to_date(date_trunc('MONTH', d))                    AS is_month_start,
  d = last_day(d)                                        AS is_month_end,
  d = to_date(date_trunc('QUARTER', d))                  AS is_quarter_start,
  d = date_sub(add_months(to_date(date_trunc('QUARTER', d)), 3), 1) AS is_quarter_end,
  (month(d) = 1 AND day(d) = 1)                          AS is_year_start,
  (month(d) = 12 AND day(d) = 31)                        AS is_year_end
FROM dates
ORDER BY d

In [0]:
%sql
SELECT
  count(*)                         AS total_rows,
  count(DISTINCT date_key)         AS distinct_date_keys,
  min(date)                        AS min_date,
  max(date)                        AS max_date,
  datediff(max(date), min(date)) + 1 AS expected_rows,
  sum(CASE WHEN is_leap_year AND month = 2 AND day_of_month = 29 THEN 1 ELSE 0 END) AS leap_days
FROM careplus_db_ws.silver.calender

In [0]:
%sql
SELECT * FROM careplus_db_ws.silver.calender
WHERE date BETWEEN DATE'2025-12-28' AND DATE'2026-01-05'
ORDER BY date